In [1]:
from discovery_child_development import PROJECT_DIR
import pandas as pd
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'
PATH_TO_DATASET = ENRICHED_DATA_DIR / 'patents_relevance_labels_only_relevant.csv'

# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

38


In [2]:
# Load the data with texts
text_df = (
    pd.read_csv(PATH_TO_DATASET)
    .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
)
len(text_df)

26178

In [3]:
dfs = []
for topic in topics:
    keywords = topics_dict[topic]["filtering_keywords"]
    df = (
        pd.read_csv(ENRICHED_DATA_DIR / f'taxonomy_cat/patents/taxonomy_cat_predictions_{topic}.csv')
        .merge(text_df[['id', 'text']], on='id', how='left')
    )
    keyword_hits = (
        df.text
        .str.lower()
        .str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)
        .str.contains("|".join(keywords))
    )
    df = df[keyword_hits]
    dfs.append(df)

In [4]:
labelled_df = (
    pd.concat(dfs, ignore_index=True)
    # Key step: taking only data that's robustly relevant
    .query("prediction==1.0")
    .groupby("id")
    .agg(topics = ("topic", list))
    .reset_index()
)

In [5]:
labelled_text_df = (
    text_df
    .merge(labelled_df, on='id', how='left')
    .set_index("id")
)

In [6]:
# Double check specific topics
extra_keywords = {
    "ai2": ["artificial intelligence", "data science", "machine learning", "deep learning", "chatbot", "natural language processing", "computer vision", "convolutional neural network", "recurrent neural network", "reinforcement learning", "predictive model", "predictive analytics"],
    "ar_vr": ["virtual reality", "augmented reality", "mixed reality"],
    "social_media": ["social media"],
    "robotics": ["robot"],
    "parenting2": ["home learning environment", "home learning", "parenting approach", "parenting style", "home learning",
    "parenting style",
    "single parent",
    "parenting skill",
    "parenting education",
    "parenting program",
    "parenting intervention",
    "parenting support",
    "parenting practice",
    "parenting behavior",
    "parenting knowledge",
    "parenting attitude",
    "parenting guidance",
    "parenting stress",
    "parent skill",
    "parent education",
    "parent program",
    "parent intervention",
    "parent support",
    "parent practice",
    "parent behavior",
    "parent knowledge",
    "parent attitude",
    "parent guidance",
    "parent stress"],
    "wearables": ["wearable", "internet of things", " iot "],
    "mobile": ["smartphone", 'ipad', 'iphone', 'android', 'phone application'],
    "infancy": ["infant", "newborn", "neonate"],
    "protection": ["child protection", "safeguarding"],
    "communication": ["language development", "speech development"],
    "cognitive": ["cognitive development"],
    "send": ["autism", "adhd", "learning disability", "special educational needs"],
    "mental_health": [" mental health "],
    "rct": ["randomised control trial", "randomized control trial"],
    "social_services": ["social service"],
    "mobile": ['mobile phone', 'smartphone', 'android', 'iphone'],
}
extra_keywords_patents = {
    "preschool": ['preschool']
}


In [7]:
for topic in extra_keywords:
    keywords = extra_keywords[topic]
    keywords = [word.lower() for word in keywords]
    hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
    hits_ids = hits_df.id.to_list()
    labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

In [8]:
for topic in extra_keywords_patents:
    keywords = extra_keywords_patents[topic]
    keywords = [word.lower() for word in keywords]
    hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
    hits_ids = hits_df.query("source == 'patents'").id.to_list()
    labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

In [9]:
text_labelled_df = (
    pd.read_csv(PATH_TO_DATASET)
    .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
    .merge(labelled_text_df.reset_index()[['id', 'topics']], on="id", how="left")
    .assign(topics = lambda df: df.topics.apply(lambda x: ", ".join(x) if type(x) == list else x))
    .drop(columns=["Unnamed: 0", "predictions"])
)

In [10]:
# Papers with no labels
n_with_topics = (text_labelled_df.topics.isnull() == False).sum()
n_without_topics = text_labelled_df.topics.isnull().sum()
n_with_topics / len(text_labelled_df), n_without_topics / len(text_labelled_df)

(0.8234013293605318, 0.17659867063946827)

In [11]:
print(n_with_topics)

21555


In [12]:
text_labelled_df.to_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_patents_filtered.csv', index=False)

## Extra checks

In [119]:
topic = "internet"
keywords = ["internet of things", " iot "]
keywords = [word.lower() for word in keywords]
hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
hits_ids = hits_df.id.to_list()
labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

hits_df_openalex = (
    hits_df
    # .query("source=='patents'")
)
topics_id = text_labelled_df[text_labelled_df.topics.astype(str).str.contains(topic)].id.to_list()
hits_df_openalex = hits_df_openalex[~hits_df_openalex.id.isin(topics_id)]
print(len(hits_df_openalex))

164


In [131]:
hits_df_openalex.sample().iloc[0].text

'System for controlliing of cradles based on voice recognition using iot. The present invention relates to a voice recognition-based smart crib control system using IoT, and the voice recognition-based smart crib control system using IoT according to an aspect of the present invention receives a motor and a control signal for controlling the motor from the outside. A communication unit and a driving unit that drives a motor based on a control signal are installed in a smart baby bed capable of swinging according to an external input, a smart baby bed, and a weight sensing unit that detects the weight of an infant on board, and recognizes a voice A voice recognition module for outputting a digital voice signal, and a voice analysis module for analyzing the frequency of the digital voice signal and outputting at least one of a first voice classification value corresponding to infant crying and a second voice classification value corresponding to an adult voice; , when the first voice cla